In [ ]:
import os
import sys

module_path = os.path.abspath(os.path.join('.'))
if module_path not in sys.path:
    sys.path.append(module_path + "/scripts/py_functions")
# import custom functions
from map_plot_tools import *
from line_plot_tools import *
from colorbar_funcs import *
from data_funcs import *
from domain_funcs import nam_domain_outline

import numpy as np
import xarray as xr
xr.set_options(keep_attrs=True)

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import matplotlib as mpl
import matplotlib.pyplot as plt
import cmocean.cm as cmo

# settings
%config InlineBackend.figure_format = 'retina'

# Observational data (OIPC, ETOPO5, IMERG). Override with OBS_DATA_DIR; see config/paths.env.example
obspath = os.environ.get('OBS_DATA_DIR', 'data/external')
# save figs here
opath = os.environ.get('FIG_OUTPUT_DIR', 'outputs')
os.makedirs(opath, exist_ok=True)

**read in data**
---
Precipitation isotope data: Online Isotopes in Precipitation Calculator<br>
source: https://wateriso.utah.edu/waterisotopes/index.html

Precipitation rate data: Integrated multi-satellite retrievals for the global precipitation measurement (GPM) mission (IMERG)<br>
source: https://gpm.nasa.gov/data/imerg <br>
reference: doi.org/10.1007/978-3-030-24568-9_19

Topography: USGS ETOPO05<br>
source:


In [ ]:
filen=f'{obspath}/obs.etopo5.zsurf.nc'
etopo_full=xr.open_dataset(f'{filen}').ROSE / 1000 # convert from m to km
etopo_full.attrs['units'] = 'km'
etopo_full=longitude_flip(etopo_full)
etopo_land=etopo_full.where(etopo_full>0, np.nan) # mask bathymetry
etopo_land = etopo_land.rename({'ETOPO05_Y':'lat','ETOPO05_X':'lon'})

In [ ]:
# set lat/lon bounds
lon_min=-125
lon_max=-85
lat_min=10
lat_max=42

filen='OIPC_monthly_data.nc'
oipc=xr.open_dataset(f'{obspath}/{filen}').isotopes[::-1,:,:].sel(Lon=slice(lon_min,lon_max), Lat=slice(lat_min,lat_max))
oipc=oipc.rename({'Lat':'lat','Lon':'lon'})
oipc=oipc.transpose('month','lat','lon')

filen='/Users/dervlamk/OneDrive/data/obs/satellite/imerg/imerg_V07_19980101-20241231_monthly_gn.nc'
imerg=xr.open_dataset(f'{filen}').precipitation.sel(lon=slice(lon_min,lon_max), lat=slice(lat_min,lat_max)) * 24 # convert from mm/hr to mm/day
del imerg.attrs["units"]
imerg.attrs['Units'] = 'mm/day'

In [ ]:
# calculate seasonal dD and prec values (assumes OIPC data is already flux-weighted)

# init dicitionaries
dD = {}
prec = {}
pprec = {}

seasons=np.array(['djf','jfm','mam','jja','jas','jjas','son','ann'])

# sum annual total precip
annual_total_p = imerg.sum(dim="time")

# get seasonal means of obs data
for season in seasons:
    months = get_season(season)
    dD[season] = oipc.isel(month=months).mean(dim='month') # isotopes
    prec[season] = imerg.isel(time=months).mean(dim='time') # precip
    
    if season != 'ann':
        pprec[season] = imerg.isel(time=months).sum(dim='time') / annual_total_p
        pprec[season].attrs['Units'] = '%'
        pprec[season].attrs['newname'] = 'percent of annual total'
    else:
        pass

In [ ]:
cmap_low = cmo.speed_r
cmap_up = cmo.turbid 
ccmap = combine_cmaps_white_center(cmap_low, cmap_up, range_low=[0,.9], range_up=[.1,.9], n_low=128, n_up=128, n_white=11)
#ccmap = combine_cmaps(cmap_low, cmap_up, range_low=[0,.825], range_up=[.15,1], n_low=128, n_up=128)
ccmap

In [ ]:
# need to fix the fact that OIPC data doesn't align with cartopy coastlines

In [ ]:
# Core locations
clons=[-106.5183, -111.62] 
clats=[22.5183, 27.85]
cnames=np.array(['NH22P','DSDP\n480/479'])
# Model Data
olon = dD['ann'].lon; olat = dD['ann'].lat
plon = prec['ann'].lon; plat = prec['ann'].lat
# settings
lw=1
arrow_kw={'arrowstyle':'-|>', 'color':'k', 'linewidth':1.25, 'shrinkB':7}
text_kw={'fontsize':12, 'fontweight':'normal', 'ha':'center'}
text_kw1={'color':'k', 'weight':'bold', 'size':16, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':18, 'ha':'center', 'va':'center'}
titles=np.array([u'Summer$-$Winter δD$_{\mathbf{prec}}$', '%Summer Precipitation'])
letters=np.array(['a','b'])
# NAM domain outline (single merged polygon, see scripts/py_functions/domain_funcs.py)
nam_domain=nam_domain_outline()
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-120., -85., 10., 37.]
tx=np.abs((map_bnds[0]-map_bnds[1])/2) + map_bnds[0]
ty=map_bnds[3]+0.5
# isotopes cmap
icmap=ccmap #cmo.balance
ivmin=-60
ivmax=60
ilevels=np.linspace(ivmin, ivmax, 41)
inorm=mpl.colors.BoundaryNorm(ilevels, icmap.N)
# precip cmap
#pcmap,_,_,_=get_settings(field='precip', diff=False)
pcmap=cmo.rain
pvmin=10
pvmax=90
plevels=np.linspace(pvmin, pvmax, 41)
pnorm=mpl.colors.BoundaryNorm(plevels, pcmap.N)


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(11,5), layout='constrained', subplot_kw={'projection': proj})
fig.text(.5,1,'', **text_kw)

#dDp
dDdiff = dD['jas'] - dD['jfm']
cf1=ax[0].pcolormesh(olon, olat, dDdiff, cmap=icmap, norm=inorm, transform=trans)
ax[0].scatter(x=clons, y=clats, c='k', alpha=1, edgecolor='k', s=150, transform=trans, zorder=100)
ax[0].annotate(cnames[1], xy=(clons[1],clats[1]), xytext=(clons[1]-5,clats[1]-6),
               arrowprops=arrow_kw, **text_kw)
ax[0].annotate(cnames[0], xy=(clons[0],clats[0]), xytext=(clons[0]-3,clats[0]-5.5),
               arrowprops=arrow_kw, **text_kw)
ax[0].add_feature(cfeature.OCEAN, fc='lightgrey')

# precip
cf2=ax[1].pcolormesh(plon, plat, pprec['jas']*100, cmap=pcmap, norm=pnorm, transform=trans)
ax[1].scatter(x=clons, y=clats, c='k', alpha=1, edgecolor='k', s=150, transform=trans, zorder=100)


for i in [0,1]:
    ax[i].contour(etopo_land.lon, etopo_land.lat, etopo_land,
              levels=np.linspace(.8, 4.8, 6), linewidths=.5, colors='k', transform=trans)
    ax[i].coastlines()
    ax[i].add_feature(cfeature.BORDERS)
    ax[i].text(tx, ty, titles[i], **text_kw1)
    ax[i].text(map_bnds[0], ty+1, letters[i], **text_kw2)
    # ONE red outline for the whole NAM domain. nam_domain_outline() unions the south and
    # north sub-domains and dissolves the edge they share, so no internal division is drawn.
    ax[i].add_geometries([nam_domain], crs=trans, fc='none', ec='r', lw=2, linestyle='--', zorder=9)
    ax[i].set_extent(map_bnds, crs=trans)
    if i==0:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False
        gl.xlabel_style = {'size': 11}
        gl.ylabel_style = {'size': 11} 
    if i==1:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.left_labels=False; gl.right_labels=False
        gl.xlabel_style = {'size': 11}

cbar_ax1 = fig.add_axes([0.055, -0.025, 0.45, 0.05])
cbar1 = fig.colorbar(cf1, orientation='horizontal', extend='both', cax=cbar_ax1)
cbar1.set_label(u'$\Delta$[‰]', size=12, labelpad=5)
cbar1.ax.tick_params(labelsize=12)

cbar_ax2 = fig.add_axes([0.54, -0.025, 0.45, 0.05])
cbar2 = fig.colorbar(cf2, ticks=[0,20,40,60,80], orientation='horizontal', extend='both', cax=cbar_ax2)
cbar2.set_label('[%]', size=12, labelpad=5)
cbar2.ax.tick_params(labelsize=12)


#fig.text(0,-0.175, r'For $\mathit{adjusted}$ (PaleoCalAdjust) iCESM1.2 LIG (127ka) output. Pattern correlation between differences for displayed domain: r $= -0.627$')
#plt.savefig(f'{opath}/modern_climo.png', dpi=1200, bbox_inches='tight')